In [4]:
# Cell 1: Load Exported Simlog Runs (Fixed Units) & Properly Aligned 60 Identification Windows
import os
import sys
import importlib
import scipy.io as sio
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

sys.path.insert(0, os.path.abspath(".."))
import export_nn_for_matlab as exporter
from export_nn_for_matlab import export_nn_for_matlab, matlab_forward

import com_physics
importlib.reload(com_physics)
from com_physics import COMPhysics

torch.set_default_dtype(torch.float64)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

FEATURES = [
    "Tamb_C", "motT", "invT", "dcdcT", "batT",
    "p_in", "p_out", "cabIntT", "cabAirT", "batI_avg"
]
if "batI_avg" not in exporter.ALLOWED:
    exporter.ALLOWED.append("batI_avg")

SIG_PATHS = {
    "T_cab":   "simlog.Cabin.Cabin_Air_Volume.T_I",
    "T_glass": "simlog.Cabin.Cabin_Heat_Transfer.Glass_Thermal_Mass.M.T",
    "T_doors": "simlog.Cabin.Cabin_Heat_Transfer.Doors_Thermal_Mass.M.T",
    "T_roof":  "simlog.Cabin.Cabin_Heat_Transfer.Roof_Thermal_Mass.M.T",
    "T_bat1":  "simlog.Battery.Pack_1.Thermal_Model.Battery_Thermal_Mass.T",
    "T_bat2":  "simlog.Battery.Pack_2.Thermal_Model.Battery_Thermal_Mass.T",
    "T_bat3":  "simlog.Battery.Pack_3.Thermal_Model.Battery_Thermal_Mass.T",
    "T_bat4":  "simlog.Battery.Pack_4.Thermal_Model.Battery_Thermal_Mass.T",
    "T_mot":   "simlog.Motor.Motor.M.T",
    "T_inv":   "simlog.Inverter.Inverter.M.T",
    "T_dcdc":  "simlog.DCDC.DC_DC_Converter.M.T",
    "w_comp":  "simlog.Compressor.Compressor.omega",
    "V_blow":  "simlog.Blower.Blower.V",
    "Q_htr":   "simlog.Heater.Controlled_Heat_Flow_Rate_Source.S",
    "Q_mot":   "simlog.Motor.Controlled_Heat_Flow_Rate_Source.S",
    "Q_inv":   "simlog.Inverter.Controlled_Heat_Flow_Rate_Source.S",
    "Q_dcdc":  "simlog.DCDC.Controlled_Heat_Flow_Rate_Source.S",
    "p_suc":   "simlog.Compressor.Compressor.A.p",
    "p_dis":   "simlog.Compressor.Compressor.B.p",
}

def causal_moving_avg(x, window=120):
    csum = np.cumsum(np.insert(x, 0, 0.0))
    idx = np.arange(len(x))
    lo = np.maximum(0, idx - window + 1)
    return (csum[idx + 1] - csum[lo]) / (idx - lo + 1)

def unpack_cells(cell_arr, N_steps):
    first_struct = next(item for item in cell_arr if hasattr(item, "_fieldnames"))
    out = {}
    for fn in first_struct._fieldnames:
        vec = np.empty(N_steps, dtype=np.float64)
        last_v = float(np.asarray(getattr(first_struct, fn)).flat[0])
        if not np.isfinite(last_v):
            last_v = 0.0
        for k in range(N_steps):
            if k < len(cell_arr) and hasattr(cell_arr[k], fn):
                val = np.asarray(getattr(cell_arr[k], fn))
                if val.size > 0 and np.isfinite(val.flat[0]):
                    last_v = float(val.flat[0])
            vec[k] = last_v
        out[fn] = vec
    return out

def load_exported_run(export_mat_path, tuning_mat_path, tamb_c):
    m = sio.loadmat(export_mat_path, squeeze_me=True)
    names = [str(n) for n in m["names"]]
    units = [str(u) for u in m["units"]]
    data = np.asarray(m["data"], dtype=np.float64)
    time_vec = np.asarray(m["time"], dtype=np.float64).ravel()

    def get_sig(key):
        path = SIG_PATHS[key]
        idx = names.index(path)
        col = data[:, idx].copy()
        u_str = units[idx].strip()
        # FIXED UNIT CONVERSION: check exact unit string first!
        if key.startswith("T_"):
            if u_str in ("degC", "C", "Celsius") or np.mean(col) < 150.0:
                col += 273.15
        elif key in ("p_suc", "p_dis"):
            if u_str == "MPa":
                col *= 1e6
            elif u_str == "bar":
                col *= 1e5
            elif u_str == "kPa":
                col *= 1e3
            elif u_str == "Pa":
                pass
            else:
                # Fallback by magnitude: MPa max is ~3.5, bar max is ~35
                if np.max(col) < 8.0:
                    col *= 1e6
                elif np.max(col) < 100.0:
                    col *= 1e5
        elif key == "w_comp":
            if u_str == "rad/s":
                col *= 60.0 / (2.0 * np.pi)
        return col

    g = {k: get_sig(k) for k in SIG_PATHS}

    S_tune = sio.loadmat(tuning_mat_path, squeeze_me=True, struct_as_record=False)
    r_tune = S_tune["resampled"]
    N_steps = min(len(time_vec), len(np.asarray(r_tune.batSocMeas).ravel()), 3600)

    batSoc  = np.asarray(r_tune.batSocMeas, dtype=np.float64).ravel()[:N_steps]
    batI    = np.asarray(r_tune.batImeas, dtype=np.float64).ravel()[:N_steps]
    motPump = np.maximum(0.0, np.asarray(r_tune.motPumpNrpm, dtype=np.float64).ravel()[:N_steps])

    T_bat = 0.25 * (g["T_bat1"] + g["T_bat2"] + g["T_bat3"] + g["T_bat4"])[:N_steps]
    T_int = ((g["T_glass"] + g["T_doors"] + g["T_roof"]) / 3.0)[:N_steps]

    X = np.column_stack([
        g["T_mot"][:N_steps], g["T_inv"][:N_steps], g["T_dcdc"][:N_steps],
        batSoc, T_bat, g["p_suc"][:N_steps], g["p_dis"][:N_steps],
        T_int, g["T_cab"][:N_steps]
    ])

    q_htr = np.maximum(0.0, g["Q_htr"][:N_steps])
    if np.max(q_htr) <= 10.0:
        q_htr *= 4000.0

    U = np.column_stack([
        np.maximum(0.0, g["w_comp"][:N_steps]),
        np.maximum(0.0, 1.225 * g["V_blow"][:N_steps]),
        motPump, motPump, q_htr, np.zeros(N_steps, dtype=np.float64)
    ])

    est_np  = unpack_cells(S_tune["sim_params"], N_steps)
    ctrl_np = unpack_cells(S_tune["controlOutCell"], N_steps)
    est_np["envT"]      = np.full(N_steps, float(tamb_c) + 273.15)
    est_np["motQmeas"]  = g["Q_mot"][:N_steps]
    est_np["invQmeas"]  = g["Q_inv"][:N_steps]
    est_np["dcdcQmeas"] = g["Q_dcdc"][:N_steps]
    est_np["batImeas"]  = batI

    batI_avg = causal_moving_avg(batI, window=120)
    F_np = np.column_stack([
        np.full(N_steps, float(tamb_c)),
        X[:, 0], X[:, 1], X[:, 2], X[:, 4],
        X[:, 5], X[:, 6], X[:, 7], X[:, 8],
        batI_avg
    ])

    return {
        "Tamb_C": tamb_c,
        "t": time_vec[:N_steps],
        "X": torch.tensor(X, device=device),
        "U": torch.tensor(U, device=device),
        "F": torch.tensor(F_np, device=device),
        "est": {k: torch.tensor(v, device=device) for k, v in est_np.items()},
        "ctrl": {k: torch.tensor(v, device=device) for k, v in ctrl_np.items()},
    }

scenarios = {
    -5:  load_exported_run("../simlog_rule_T5_export.mat",  "../Data/tuning_data_5.mat",  -5.0),
    -7:  load_exported_run("../simlog_rule_T7_export.mat",  "../Data/tuning_data_7.mat",  -7.0),
    -10: load_exported_run("../simlog_rule_T10_export.mat", "../Data/tuning_data_10.mat", -10.0),
}

print(f"Sanity check on T10 pressures: p_in mean = {scenarios[-10]['X'][:, 5].mean().item()/1e5:.2f} bar, "
      f"p_out mean = {scenarios[-10]['X'][:, 6].mean().item()/1e5:.2f} bar")

# Load the 60 identification windows paired with their exact identification states (23°C reference)
Stuned = sio.loadmat("../Data/tunedVariables_minus5_minus10.mat", squeeze_me=True, struct_as_record=False)
TV = Stuned["tunedVariables"]
P_all = np.atleast_2d(np.asarray(TV.parameters, dtype=np.float64))
A_all = np.asarray(TV.ambient, dtype=np.float64).ravel()
wl = int(np.asarray(TV.meta.windowLength).ravel()[0])

win_F_list, win_P_list = [], []
for tamb, fname in [(-10, "../Data/tuning_data_10.mat"), (-5, "../Data/tuning_data_5.mat")]:
    r = sio.loadmat(fname, squeeze_me=True, struct_as_record=False)["resampled"]
    g = lambda n: np.asarray(getattr(r, n), dtype=np.float64).ravel()
    X_id = np.column_stack([
        g("motTmeasC") + 273.15, g("invTmeasC") + 273.15, g("dcdcTmeasC") + 273.15,
        g("batSocMeas"), g("batTmeasC") + 273.15, g("compPinMpa") * 1e6,
        g("compPoutMpa") * 1e6, g("cabIntTInK"), g("cabAirTmeasC") + 273.15
    ])
    I_id = g("batImeas")
    P_temp = P_all[A_all == tamb]
    n_win = len(P_temp) // wl
    for j in range(n_win):
        sl = slice(j * wl, (j + 1) * wl)
        xm = X_id[sl].mean(axis=0)
        f_row = [float(tamb), xm[0], xm[1], xm[2], xm[4], xm[5], xm[6], xm[7], xm[8], I_id[sl].mean()]
        win_F_list.append(f_row)
        win_P_list.append(P_temp[sl][0])

win_F = np.array(win_F_list)
win_P = np.array(win_P_list)
p_lo = win_P.min(axis=0)
p_hi = win_P.max(axis=0)
print(f"✓ Loaded {len(win_F)} properly aligned identification windows and 3 exported simlog runs!")

Using device: cpu
Sanity check on T10 pressures: p_in mean = 2.06 bar, p_out mean = 34.90 bar
✓ Loaded 60 properly aligned identification windows and 3 exported simlog runs!


In [5]:
# Cell 2: Stage 1 (10-Member Ensemble Warm-Start) + Stage 2 (End-to-End COMPhysics Refinement)
com = COMPhysics().to(device)

# Compute normalization stats from the 60 identification windows
x_mean_np = win_F.mean(axis=0)
x_std_np  = win_F.std(axis=0)
x_std_np[x_std_np < 1e-9] = 1.0

log_win_P = np.log(np.maximum(win_P, 1e-6))
y_mean_np = log_win_P.mean(axis=0)
y_std_np  = log_win_P.std(axis=0)
y_std_np[y_std_np < 1e-9] = 1.0

x_mean = torch.tensor(x_mean_np, device=device)
x_std  = torch.tensor(x_std_np, device=device)
y_mean = torch.tensor(y_mean_np, device=device)
y_std  = torch.tensor(y_std_np, device=device)
p_lo_t = torch.tensor(p_lo, device=device)
p_hi_t = torch.tensor(p_hi, device=device)

# Down-weight boundary-clamped NLS labels (0.2 weight)
span = p_hi - p_lo
n_unique = np.array([len(np.unique(np.round(win_P[:, i], 9))) for i in range(11)])
graded = (span > 1e-9) & (n_unique > 2)
at_bound = (np.isclose(win_P, p_lo) | np.isclose(win_P, p_hi)) & graded
W_win = torch.tensor(np.where(at_bound, 0.2, 1.0), device=device)

F_win_norm = torch.tensor((win_F - x_mean_np) / x_std_np, device=device)
Y_win_norm = torch.tensor((log_win_P - y_mean_np) / y_std_np, device=device)

# -------------------------------------------------------------------------
# STAGE 1: Train Ensemble of 10 Networks & Merge into Single Layer
# -------------------------------------------------------------------------
HIDDEN = 8
ENSEMBLE = 10
sub_models = []

for e in range(ENSEMBLE):
    torch.manual_seed(100 + e)
    m = nn.Sequential(
        nn.Linear(len(FEATURES), HIDDEN),
        nn.Tanh(),
        nn.Linear(HIDDEN, 11)
    ).to(device)
    opt = torch.optim.Adam(m.parameters(), lr=1e-2, weight_decay=3e-2)
    for _ in range(3000):
        loss = (W_win * (m(F_win_norm) - Y_win_norm) ** 2).sum() / W_win.sum()
        opt.zero_grad()
        loss.backward()
        opt.step()
    sub_models.append(m)

# Merge the 10 ensemble members into one equivalent 80-neuron 1-hidden-layer network
merged_net = nn.Sequential(
    nn.Linear(len(FEATURES), ENSEMBLE * HIDDEN),
    nn.Tanh(),
    nn.Linear(ENSEMBLE * HIDDEN, 11)
).to(device)

with torch.no_grad():
    merged_net[0].weight.copy_(torch.cat([m[0].weight for m in sub_models], dim=0))
    merged_net[0].bias.copy_(torch.cat([m[0].bias for m in sub_models], dim=0))
    merged_net[2].weight.copy_(torch.cat([m[2].weight for m in sub_models], dim=1) / ENSEMBLE)
    merged_net[2].bias.copy_(torch.stack([m[2].bias for m in sub_models], dim=0).mean(dim=0))

def forward_exact_matlab(net, f_raw):
    """100% identical to matlab_forward() in export_nn_for_matlab.py and get_tuneS.m."""
    z = (f_raw - x_mean) / x_std
    raw_out = net(z) * y_std + y_mean
    p = torch.exp(raw_out)
    return torch.clamp(p, p_lo_t, p_hi_t)

with torch.no_grad():
    for tamb in [-5, -7, -10]:
        scenarios[tamb]["P_stage1"] = forward_exact_matlab(merged_net, scenarios[tamb]["F"]).detach()
print("✓ Stage 1 Ensemble Merged Network initialized!")

# -------------------------------------------------------------------------
# STAGE 2: End-to-End COMPhysics Fine-Tuning on Exported Simlog Trajectories
# -------------------------------------------------------------------------
import copy
best_state_dict = copy.deepcopy(merged_net.state_dict())

def eval_30s_rollout_loss(net, temp_list):
    net.eval()
    H = 30
    weights = torch.tensor([1.0, 1.0, 0.5, 0.0, 2.0, 0.0, 0.0, 0.5, 4.0], device=device)
    total_err = 0.0
    with torch.no_grad():
        for tamb in temp_list:
            sc = scenarios[tamb]
            T_len = sc["X"].shape[0] - H
            P_all_k = forward_exact_matlab(net, sc["F"][:T_len])
            x_sim = sc["X"][:T_len].clone()
            for i in range(H):
                u_i = sc["U"][i : i + T_len]
                est_i = {k: v[i : i + T_len] for k, v in sc["est"].items()}
                ctrl_i = {k: v[i : i + T_len] for k, v in sc["ctrl"].items()}
                x_sim = com.step_rk4(x_sim, u_i, est_i, ctrl_i, P_all_k, dt=1.0)
                x_true = sc["X"][i + 1 : i + 1 + T_len]
                total_err += torch.mean(weights * ((x_sim - x_true) ** 2)).item() / H
    return total_err / len(temp_list)

base_val_loss = eval_30s_rollout_loss(merged_net, [-7])
best_val_loss = base_val_loss
print(f"Initial Stage 1 Unseen -7°C 30s Rollout Weighted MSE: {base_val_loss:.5f}")

H_ROLL = 30
state_weights = torch.tensor([1.0, 1.0, 0.5, 0.0, 2.0, 0.0, 0.0, 0.5, 4.0], device=device)
opt_e2e = torch.optim.Adam(merged_net.parameters(), lr=1e-4)
rng = np.random.default_rng(42)

pbar = tqdm(range(1, 61), desc="Stage 2: End-to-End COMPhysics Fine-Tuning")
for epoch in pbar:
    merged_net.train()
    opt_e2e.zero_grad()
    loss_total = 0.0

    for tamb in [-5, -10]:
        sc = scenarios[tamb]
        max_start = sc["X"].shape[0] - H_ROLL - 1
        starts = torch.tensor(rng.integers(0, max_start, size=64), dtype=torch.long, device=device)

        # Use the exact MATLAB forward formula so there is zero export mismatch!
        P_k = forward_exact_matlab(merged_net, sc["F"][starts])
        x_sim = sc["X"][starts]

        roll_loss = 0.0
        for i in range(H_ROLL):
            idx = starts + i
            u_i = sc["U"][idx]
            est_i = {k: v[idx] for k, v in sc["est"].items()}
            ctrl_i = {k: v[idx] for k, v in sc["ctrl"].items()}
            x_sim = com.step_rk4(x_sim, u_i, est_i, ctrl_i, P_k, dt=1.0)
            x_true = sc["X"][idx + 1]
            roll_loss = roll_loss + torch.mean(state_weights * ((x_sim - x_true) ** 2)) / H_ROLL

        # Strong anchor to Stage 1 ensemble + 60 window labels so parameters never drift
        P_anchor = sc["P_stage1"][starts]
        anchor_loss = 5.0 * torch.mean(((torch.log(P_k) - torch.log(P_anchor)) / y_std) ** 2)
        win_loss = 2.0 * (W_win * (merged_net(F_win_norm) - Y_win_norm) ** 2).sum() / W_win.sum()

        loss_total = loss_total + roll_loss + anchor_loss + win_loss

    loss_total.backward()
    torch.nn.utils.clip_grad_norm_(merged_net.parameters(), 0.5)
    opt_e2e.step()

    if epoch % 10 == 0:
        val_loss = eval_30s_rollout_loss(merged_net, [-7])
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_state_dict = copy.deepcopy(merged_net.state_dict())
        pbar.set_postfix({"val_mse": f"{val_loss:.5f}", "best": f"{best_val_loss:.5f}"})

# Restore the best validated checkpoint
merged_net.load_state_dict(best_state_dict)
print(f"✓ Restored best checkpoint (Unseen -7°C 30s Rollout MSE: {best_val_loss:.5f} vs Stage 1: {base_val_loss:.5f})")

✓ Stage 1 Ensemble Merged Network initialized!
Initial Stage 1 Unseen -7°C 30s Rollout Weighted MSE: 0.09912


Stage 2: End-to-End COMPhysics Fine-Tuning:   0%|          | 0/60 [00:00<?, ?it/s]

✓ Restored best checkpoint (Unseen -7°C 30s Rollout MSE: 0.09291 vs Stage 1: 0.09912)


In [6]:
# Cell 3: Export ../nn_anton.mat and Verify 100% Match with MATLAB
f_span = win_F.max(axis=0) - win_F.min(axis=0)
env_lo = win_F.min(axis=0) - 0.25 * f_span
env_hi = win_F.max(axis=0) + 0.25 * f_span

out_mat_path = "../nn_anton.mat"
export_nn_for_matlab(
    merged_net,
    features=FEATURES,
    x_mean=x_mean_np,
    x_std=x_std_np,
    path=out_mat_path,
    y_mean=y_mean_np,
    y_std=y_std_np,
    out_transform="exp",
    p_lo=p_lo,
    p_hi=p_hi,
    env_lo=env_lo,
    env_hi=env_hi
)

# Verify exact numerical agreement with MATLAB forward pass
d = sio.loadmat(out_mat_path, squeeze_me=False)
d = {k: (v if not isinstance(v, np.ndarray) or v.dtype.kind != "U" else str(v.ravel()[0]))
     for k, v in d.items() if not k.startswith("__")}
d["act"] = "tanh"
d["out_transform"] = "exp"
d["n_layers"] = float(np.asarray(d["n_layers"]).ravel()[0])

with torch.no_grad():
    py_preds = forward_exact_matlab(merged_net, torch.tensor(win_F, device=device)).cpu().numpy()
    mat_preds = np.array([matlab_forward(d, row) for row in win_F])
    max_err = np.max(np.abs(py_preds - mat_preds))

print(f"✓ MATLAB forward pass verification error: {max_err:.2e}")
print("✓ Now run in MATLAB:  validate_com_fig2({'interp', 'anton'})")

Exported ../nn_anton.mat: inputs ['Tamb_C', 'motT', 'invT', 'dcdcT', 'batT', 'p_in', 'p_out', 'cabIntT', 'cabAirT', 'batI_avg'], 2 layers, act=tanh
✓ MATLAB forward pass verification error: 5.33e-15
✓ Now run in MATLAB:  validate_com_fig2({'interp', 'anton'})
